# **DATA PULL, CLEANING AND VISUALIZATION**

## EDA

### **Data Pull**
This code pulls up confirmed exoplanet data from the NASA Exoplanet Archive's public TAP (Table Access Protocol) API and saves it as a raw CSV for the
DataPrep_EDA tab of my project.

> On githuhb repo - # fetch_exoplanet_data.py

> Also downloaded the data from -

https://phl.upr.edu/hwc/data



In [1]:
# Impoting Libraries.
import pandas as pd
import requests

In [2]:
# TAP STuff - table access protocol (like API request for astronomy)
# what I'm doing after this "requests.get(TAP_URL, params=params)"- This is an HTTP request to a server, with structured parameters,
# returning structured data back (CSV) that the code parses programmatically.
# That's precisely what an API is: a defined way for code to talk to a server and get data back, no browser, no manual download, no clicking through a webpage.
TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

In [3]:
# Columns from the Planetary Systems Composite Parameters table
# (pscomppars): one best estimate row per confirmed planet.
COLUMNS = [
    "pl_name",        # planet name
    "hostname",       # host star name
    "discoverymethod",# transit, radial velocity, imaging, microlensing, etc. etc.
    "disc_year",      # year of discovery
    "pl_orbper",      # orbital period (days)
    "pl_rade",        # planet radius (Earth radii)
    "pl_bmasse",      # planet mass (Earth masses)
    "pl_eqt",         # equilibrium temperature (K)
    "st_teff",        # host star effective temperature (K)
    "st_rad",         # host star radius (solar radii)
    "st_mass",        # host star mass (solar masses)
    "sy_dist",        # system distance from Earth (parsecs)
]


In [4]:
def fetch_raw_data() -> pd.DataFrame:
    query = f"select {','.join(COLUMNS)} from pscomppars"
    params = {"query": query, "format": "csv"}
    response = requests.get(TAP_URL, params=params, timeout=60)
    response.raise_for_status()

# requests.Response has .url with the final, fully-encoded GET URL.
# This is the exact request in the DataPrep_EDA section on my site.
    print("Request URL:", response.url)

    from io import StringIO
    return pd.read_csv(StringIO(response.text))

if __name__ == "__main__":
    df = fetch_raw_data()
    df.to_csv("exoplanets_raw.csv", index=False)
    print(f"Saved {len(df)} rows to exoplanets_raw.csv")
    print(df.head())

Request URL: https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+pl_name%2Chostname%2Cdiscoverymethod%2Cdisc_year%2Cpl_orbper%2Cpl_rade%2Cpl_bmasse%2Cpl_eqt%2Cst_teff%2Cst_rad%2Cst_mass%2Csy_dist+from+pscomppars&format=csv
Saved 6451 rows to exoplanets_raw.csv
    pl_name hostname  discoverymethod  disc_year    pl_orbper  pl_rade  \
0  55 Cnc b   55 Cnc  Radial Velocity       1996    14.651552   13.900   
1  55 Cnc c   55 Cnc  Radial Velocity       2004    44.393600    8.510   
2  55 Cnc d   55 Cnc  Radial Velocity       2002  4799.000000   13.000   
3  55 Cnc e   55 Cnc  Radial Velocity       2004     0.736547    1.875   
4  55 Cnc f   55 Cnc  Radial Velocity       2007   260.580000    7.590   

   pl_bmasse   pl_eqt  st_teff  st_rad  st_mass  sy_dist  
0   263.9785   700.00   5198.0   0.980    1.015  12.5855  
1    54.4738   499.27   5198.0   0.980    1.015  12.5855  
2  1232.4930   104.85   5198.0   0.980    1.015  12.5855  
3     7.9900  1958.00   5172.0   0.943    0.90

### **CLEANING AND VISUALIZATION**

 > On githuhb repo - # clean_and_visualize.py

Cleans the raw NASA Exoplanet Archive pull, merges in habitability labels from the PHL Habitable Worlds Catalog and produces the 11 visualizations and two data-preview images used on the Cleaning & Prep and EDA tabs.

Inputs:

    exoplanets_raw.csv   from fetch_exoplanet_data.py

    hwc.csv              the "Full Catalog (CSV)" downloaded from phl.upr.edu/hwc/data

    hwc_table_all.csv    the 70-row shortlist table (Table 1 + Table 2) from the same page, saved as CSV

Outputs:

    exoplanets_clean.csv       cleaned + merged dataset (20 columns)

    charts/01_missingness.png ... charts/12_top_esi_table.png

    charts/raw_preview.png, charts/clean_preview.png

In [5]:
# Importing more libraries -
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [6]:
# the theme to match the site's CSS variables in style.css # Looked at documentations etc. for this one.
BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "grid.color": LINE,
    "font.size": 11, "axes.titlesize": 13, "axes.titlecolor": TEXT,})

os.makedirs("charts", exist_ok=True)

df = pd.read_csv("exoplanets_raw.csv")

In [7]:
# CLEANING
clean = df.copy()
# checking for duplicate planet names
print("Duplicate planet names:", clean["pl_name"].duplicated().sum())

# checking for negative or zero values in the numeric columns, which would be
# physically impossible for these measurements
numeric_cols = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]
for col in numeric_cols:
    n_bad = (clean[col] <= 0).sum()
    print(f"{col}: {n_bad} values <= 0")

# checking how much is missing in each column
print("\nMissing values per column:")
print(clean.isna().sum())

# 157 planets sit above the ~13 Jupiter mass (4131 Earth mass) boundary,
# where "planet" starts overlapping with "brown dwarf" - flagged rather
# than dropped, since they are still confirmed entries in the archive
clean["is_borderline_massive"] = clean["pl_bmasse"] > 4131
print("\nBorderline massive planets flagged:", clean["is_borderline_massive"].sum())

Duplicate planet names: 0
pl_orbper: 0 values <= 0
pl_rade: 0 values <= 0
pl_bmasse: 0 values <= 0
pl_eqt: 0 values <= 0
st_teff: 0 values <= 0
st_rad: 0 values <= 0
st_mass: 0 values <= 0
sy_dist: 0 values <= 0

Missing values per column:
pl_name              0
hostname             0
discoverymethod      0
disc_year            0
pl_orbper          354
pl_rade             50
pl_bmasse           30
pl_eqt             540
st_teff            307
st_rad             332
st_mass              9
sy_dist             28
dtype: int64

Borderline massive planets flagged: 157


- Duplicate planet names and no negative/zero physical values were found in any numeric column on inspection, so nothing needed to be dropped on those grounds.

- Missingness is left as NaN rather than imputed here, since the right way to fill it depends on which model uses that column later.

- Flag planets above the ~13-Jupiter-mass (4131 Earth-mass)boundary, where "planet" starts overlapping with "brown dwarf" by common convention.

- Kept (they are confirmed in the archive) but flagged rather than treated as ordinary planets.
- As for the plots: JUST saving my images and charts in the back instead of plt.show() here.

In [8]:
# Discretize planet size into standard exoplanet-science size classes.
def size_class(r):
    if pd.isna(r): return np.nan
    if r < 1.25: return "Earth-sized"
    if r < 2.0: return "Super-Earth"
    if r < 6.0: return "Sub-Neptune/Neptune"
    return "Giant"
clean["size_class"] = clean["pl_rade"].apply(size_class)

# Discretize equilibrium temperature into a rough habitability zone.
# ~200-320 K is the loose range where liquid water is plausible.
def temp_zone(t):
    if pd.isna(t): return np.nan
    if t < 200: return "Too Cold"
    if t <= 320: return "Temperate"
    return "Too Hot"
clean["temp_zone"] = clean["pl_eqt"].apply(temp_zone)

In [9]:
# MERGE WITH PHL HABITABLE WORLDS CATALOG
hwc = pd.read_csv("hwc.csv")
hwc_cols = hwc[["P_NAME", "P_HABITABLE", "P_ESI", "P_TYPE",
                "P_HABZONE_OPT", "P_HABZONE_CON"]].rename(columns={"P_NAME": "pl_name"})

clean = clean.merge(hwc_cols, on="pl_name", how="left")
clean["hwc_habitable_label"] = clean["P_HABITABLE"].map(
    {0: "Not habitable", 1: "Conservative sample", 2: "Optimistic sample"})

clean.to_csv("exoplanets_clean.csv", index=False)
print("Cleaned + merged file saved:", clean.shape)
print(clean["size_class"].value_counts(dropna=False))
print(clean["temp_zone"].value_counts(dropna=False))
print(clean["hwc_habitable_label"].value_counts(dropna=False))

# PREVIEW TABLE IMAGES
def save_table_image(d, cols, fname, title):
    sample = d[cols].head(6)
    fig, ax = plt.subplots(figsize=(11, 2.2))
    ax.axis("off")
    ax.set_title(title, loc="left", color=GOLD, fontsize=12, pad=14)
    tbl = ax.table(cellText=sample.round(2).astype(str).values,
                    colLabels=cols, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.6)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor(LINE)
        if row == 0:
            cell.set_facecolor(PANEL)
            cell.set_text_props(color=GOLD, weight="bold")
        else:
            cell.set_facecolor(BG)
            cell.set_text_props(color=TEXT)
    plt.savefig(f"charts/{fname}", dpi=150, bbox_inches="tight")
    plt.close()

save_table_image(df, ["pl_name", "hostname", "discoverymethod", "disc_year", "pl_rade", "pl_bmasse"],
                  "raw_preview.png", "Raw data (as pulled from the API)")
save_table_image(clean, ["pl_name", "pl_rade", "size_class", "pl_eqt", "temp_zone", "is_borderline_massive"],
                  "clean_preview.png", "Cleaned data (derived columns added)")

Cleaned + merged file saved: (6451, 21)
size_class
Sub-Neptune/Neptune    2442
Giant                  2164
Super-Earth            1212
Earth-sized             583
NaN                      50
Name: count, dtype: int64
temp_zone
Too Hot      5219
NaN           540
Temperate     363
Too Cold      329
Name: count, dtype: int64
hwc_habitable_label
Not habitable          5495
NaN                     886
Optimistic sample        41
Conservative sample      29
Name: count, dtype: int64


### **VISUALIZATIONS**

Plotting and saving charts in the backend

In [10]:
# VISUALIZATIONS
# JUST saving my images and charts in the back instead of plt.show() here.
# 1. Missingness per column
miss = df.isna().mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(miss.index, miss.values, color=TEAL)
ax.set_xlabel("% missing")
ax.set_title("Missing data by column")
ax.invert_yaxis()
plt.tight_layout(); plt.savefig("charts/01_missingness.png", dpi=150); plt.close()

# 2. Discovery method counts (log scale)
dm = df["discoverymethod"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(dm.index[::-1], dm.values[::-1], color=GOLD)
ax.set_xscale("log")
ax.set_xlabel("Number of confirmed planets (log scale)")
ax.set_title("Confirmed planets by discovery method")
plt.tight_layout(); plt.savefig("charts/02_discovery_method.png", dpi=150); plt.close()

# 3. Discoveries per year
yr = df["disc_year"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(yr.index, yr.values, color=RUST, linewidth=2)
ax.fill_between(yr.index, yr.values, color=RUST, alpha=0.15)
ax.set_xlabel("Year"); ax.set_ylabel("Planets confirmed")
ax.set_title("Confirmed exoplanet discoveries per year")
plt.tight_layout(); plt.savefig("charts/03_discoveries_per_year.png", dpi=150); plt.close()

# 4. Planet radius distribution (log)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["pl_rade"].dropna(), bins=40, color=TEAL)
ax.set_xscale("log")
ax.set_xlabel("Planet radius (Earth radii, log scale)"); ax.set_ylabel("Count")
ax.set_title("Distribution of planet radius")
plt.tight_layout(); plt.savefig("charts/04_radius_distribution.png", dpi=150); plt.close()

# 5. Planet mass distribution (log)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["pl_bmasse"].dropna(), bins=40, color=GOLD)
ax.set_xscale("log")
ax.set_xlabel("Planet mass (Earth masses, log scale)"); ax.set_ylabel("Count")
ax.set_title("Distribution of planet mass")
plt.tight_layout(); plt.savefig("charts/05_mass_distribution.png", dpi=150); plt.close()

# 6. Mass vs radius scatter
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["pl_rade"], df["pl_bmasse"], s=8, alpha=0.4, color=TEAL)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Radius (Earth radii, log)"); ax.set_ylabel("Mass (Earth masses, log)")
ax.set_title("Planet mass vs. radius")
plt.tight_layout(); plt.savefig("charts/06_mass_vs_radius.png", dpi=150); plt.close()

# 7. Orbital period vs equilibrium temperature
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["pl_orbper"], df["pl_eqt"], s=8, alpha=0.4, color=RUST)
ax.set_xscale("log")
ax.set_xlabel("Orbital period (days, log scale)"); ax.set_ylabel("Equilibrium temperature (K)")
ax.set_title("Orbital period vs. equilibrium temperature")
plt.tight_layout(); plt.savefig("charts/07_period_vs_temp.png", dpi=150); plt.close()

# 8. Host star temperature histogram
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["st_teff"].dropna(), bins=40, color=GOLD)
ax.axvline(5778, color=TEXT, linestyle="--", linewidth=1, label="Sun (5778 K)")
ax.set_xlabel("Host star effective temperature (K)"); ax.set_ylabel("Count")
ax.set_title("Host star temperature")
ax.legend()
plt.tight_layout(); plt.savefig("charts/08_star_temperature.png", dpi=150); plt.close()

# 9. Size class breakdown
sc = clean["size_class"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(sc.index, sc.values, color=[TEAL, GOLD, RUST, DIM])
ax.set_ylabel("Count")
ax.set_title("Planets by size class")
plt.tight_layout(); plt.savefig("charts/09_size_class.png", dpi=150); plt.close()

# 10. Temperature zone breakdown
tz = clean["temp_zone"].value_counts().reindex(["Too Cold", "Temperate", "Too Hot"])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(tz.index, tz.values, color=[TEAL, GOLD, RUST])
ax.set_ylabel("Count")
ax.set_title("Planets by equilibrium-temperature zone")
plt.tight_layout(); plt.savefig("charts/10_temp_zone.png", dpi=150); plt.close()

# 11. Official HWC habitability classification
counts = clean["hwc_habitable_label"].value_counts().reindex(
    ["Not habitable", "Conservative sample", "Optimistic sample"]
)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(counts.index, counts.values, color=[DIM, TEAL, GOLD])
ax.set_ylabel("Count")
ax.set_title("Official Habitable Worlds Catalog classification")
for i, v in enumerate(counts.values):
    ax.text(i, v + 40, str(int(v)), ha="center", color=TEXT, fontsize=10)
plt.tight_layout(); plt.savefig("charts/11_hwc_habitability.png", dpi=150); plt.close()

# 12. Top 10 most Earth-like worlds by ESI, from the shortlist table
short = pd.read_csv("hwc_table_all.csv")
short.columns = [re.sub(r"<[^>]+>", "", c) for c in short.columns]
short = short.sort_values("ESI", ascending=False).head(10)
short_display = short[["Name", "Type", "ESI"]].round({"ESI": 3})

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.axis("off")
ax.set_title("Top 10 potentially habitable worlds, by Earth Similarity Index",
             loc="left", color=GOLD, fontsize=12, pad=14)
tbl = ax.table(cellText=short_display.values, colLabels=["Name", "Type", "ESI"],
                cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(LINE)
    if row == 0:
        cell.set_facecolor(PANEL); cell.set_text_props(color=GOLD, weight="bold")
    else:
        cell.set_facecolor(BG); cell.set_text_props(color=TEXT)
plt.savefig("charts/12_top_esi_table.png", dpi=150, bbox_inches="tight")
plt.close()

print("All charts saved to charts/")

All charts saved to charts/


# **MODULE 2**
## Clustering and PCA

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

In [12]:
os.makedirs("charts", exist_ok=True)

df = pd.read_csv("exoplanets_clean.csv")

# DATA PREP
# Clustering and PCA need unlabeled numeric data only, so just the eight
# physical measurement columns are used here, with rows missing any of
# them dropped.
numeric_cols = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt",
                 "st_teff", "st_rad", "st_mass", "sy_dist"]
cluster_df = df.dropna(subset=numeric_cols).reset_index(drop=True)
print("Rows with complete numeric data:", len(cluster_df), "of", len(df))

# these columns span several orders of magnitude (orbital period especially),
# so log-transform first so no single feature dominates the distance purely
# because of its raw scale, then standardize to mean 0 / std 1
X_log = np.log10(cluster_df[numeric_cols])
X_log.columns = [c + "_log" for c in numeric_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

#  K-MEANS
silhouette_by_k = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_by_k[k] = silhouette_score(X_scaled, labels)
    print(f"k={k}: silhouette={silhouette_by_k[k]:.4f}")

best_k = max(silhouette_by_k, key=silhouette_by_k.get)
print("Best k by silhouette:", best_k)

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = km_best.fit_predict(X_scaled)

# silhouette score vs k
ks = sorted(silhouette_by_k.keys())
vals = [silhouette_by_k[k] for k in ks]
plt.figure(figsize=(7, 4.5))
plt.plot(ks, vals, marker="o")
plt.xlabel("k (number of clusters)")
plt.ylabel("Average silhouette score")
plt.title("K-means silhouette score by k")
plt.xticks(ks)
plt.tight_layout()
plt.savefig("charts/silhouette_by_k.png", dpi=150)
plt.close()

# PCA (computed early so clusters can be visualized in PC space)
pca = PCA()
pcs = pca.fit_transform(X_scaled)
evr = pca.explained_variance_ratio_
print("Explained variance ratio:", evr)
print("Cumulative (first 2):", evr[:2].sum())

loadings = pd.DataFrame(pca.components_[:2].T, index=X_log.columns, columns=["PC1", "PC2"])
print(loadings.round(3))

# k-means clusters shown in PC1/PC2 space
plt.figure(figsize=(7, 6))
scatter = plt.scatter(pcs[:, 0], pcs[:, 1], c=kmeans_labels, s=8, alpha=0.5, cmap="tab10")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"K-means clusters (k={best_k}), shown in PC1/PC2 space")
plt.legend(*scatter.legend_elements(), title="Cluster", fontsize=8)
plt.tight_layout()
plt.savefig("charts/kmeans_clusters_pca.png", dpi=150)
plt.close()

# HIERARCHICAL CLUSTERING (cosine distance)
# cosine distance can't be combined with ward/centroid linkage, so average
# linkage is used instead
dist = pdist(X_scaled, metric="cosine")
Z = linkage(dist, method="average")

plt.figure(figsize=(9, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=8, show_contracted=True)
plt.xlabel("Cluster (count of original points in parentheses)")
plt.ylabel("Cosine distance")
plt.title("Hierarchical clustering dendrogram (average linkage, cosine distance)")
plt.tight_layout()
plt.savefig("charts/dendrogram.png", dpi=150)
plt.close()

# reading the dendrogram's own suggested k from the biggest jump in merge height
# among the last several merges
last_merges = Z[-10:, 2]
diffs = np.diff(last_merges)
jump_idx = np.argmax(diffs)
suggested_k = len(last_merges) - jump_idx
print("Dendrogram-suggested k (largest merge-height jump):", suggested_k)

# comparing k-means and hierarchical clustering at the same k
hclust_labels = fcluster(Z, t=best_k, criterion="maxclust")
ari = adjusted_rand_score(kmeans_labels, hclust_labels)
print(f"Adjusted Rand Index, k-means vs hierarchical at k={best_k}:", round(ari, 4))

contingency = pd.crosstab(pd.Series(kmeans_labels, name="kmeans"),
                           pd.Series(hclust_labels, name="hclust"))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(contingency.values, cmap="Blues")
ax.set_xticks(range(len(contingency.columns))); ax.set_xticklabels(contingency.columns)
ax.set_yticks(range(len(contingency.index))); ax.set_yticklabels(contingency.index)
ax.set_xlabel("Hierarchical cluster"); ax.set_ylabel("K-means cluster")
ax.set_title("How K-means and hierarchical clusters overlap")
for i in range(contingency.shape[0]):
    for j in range(contingency.shape[1]):
        ax.text(j, i, contingency.values[i, j], ha="center", va="center", fontsize=9)
plt.colorbar(im, label="Number of planets")
plt.tight_layout()
plt.savefig("charts/kmeans_vs_hclust.png", dpi=150)
plt.close()

# PCA VISUALIZATIONS
plt.figure(figsize=(7, 4.5))
plt.bar(range(1, len(evr) + 1), evr)
plt.plot(range(1, len(evr) + 1), np.cumsum(evr), marker="o", color="black", label="Cumulative")
plt.xlabel("Principal component"); plt.ylabel("Proportion of variance explained")
plt.title("PCA scree plot")
plt.legend()
plt.tight_layout()
plt.savefig("charts/pca_scree.png", dpi=150)
plt.close()

plt.figure(figsize=(7, 6))
plt.scatter(pcs[:, 0], pcs[:, 1], s=6, alpha=0.3)
plt.xlabel(f"PC1 ({evr[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({evr[1]*100:.1f}% variance)")
plt.title("Exoplanets projected onto the first two principal components")
plt.tight_layout()
plt.savefig("charts/pca_pc1_pc2.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pcs[:, 0], pcs[:, 1], s=5, alpha=0.15, color="gray")
for feat in loadings.index:
    x, y = loadings.loc[feat, "PC1"] * 6, loadings.loc[feat, "PC2"] * 6
    ax.arrow(0, 0, x, y, color="red", width=0.02, head_width=0.15)
    ax.text(x * 1.1, y * 1.1, feat.replace("_log", ""), color="red", fontsize=9, ha="center")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("PCA loadings: which features drive PC1 and PC2")
plt.tight_layout()
plt.savefig("charts/pca_loadings.png", dpi=150)
plt.close()

print("Module 2 charts saved to charts/")

Rows with complete numeric data: 5756 of 6451
k=2: silhouette=0.3283
k=3: silhouette=0.3620
k=4: silhouette=0.3488
k=5: silhouette=0.3633
k=6: silhouette=0.3192
k=7: silhouette=0.2673
Best k by silhouette: 5
Explained variance ratio: [0.40091891 0.32091814 0.12531226 0.07037671 0.0558116  0.01283792
 0.00994071 0.00388376]
Cumulative (first 2): 0.7218370422335827
                 PC1    PC2
pl_orbper_log  0.271 -0.434
pl_rade_log    0.421 -0.248
pl_bmasse_log  0.423 -0.282
pl_eqt_log     0.016  0.533
st_teff_log    0.357  0.318
st_rad_log     0.456  0.113
st_mass_log    0.473  0.254
sy_dist_log    0.106  0.456
Dendrogram-suggested k (largest merge-height jump): 2
Adjusted Rand Index, k-means vs hierarchical at k=5: 0.5196
Module 2 charts saved to charts/


## Module 3 data prep: building the labeled dataset and split it into training and testing sets.
#### Input: exoplanets_clean.csv (from clean_and_visualize.py)
#### Output: labeled_sample.csv, train_set.csv, test_set.csv, plus small preview images in charts/

In [13]:
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# keeping the dark bg going :)
BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "font.size": 11,
    "axes.titlesize": 13, "axes.titlecolor": TEXT})

os.makedirs("charts", exist_ok=True)
df = pd.read_csv("exoplanets_clean.csv")

features = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]

# supervised methods need a label. The label here comes from the PHL Habitable Worlds
# Catalog: P_HABITABLE is 0 (not habitable), 1 (conservative sample) or 2 (optimistic sample).
# Collapsed into one yes/no label: habitable = 1 if the catalog lists the planet in either sample.
# Only planets the catalog has assessed (P_HABITABLE present) and with all 8 measurements are kept.
labeled = df.dropna(subset=features + ["P_HABITABLE"]).copy()
labeled["habitable"] = (labeled["P_HABITABLE"] > 0).astype(int)
labeled = labeled[["pl_name"] + features + ["habitable"]].reset_index(drop=True)
print("Labeled rows:", len(labeled))
print(labeled["habitable"].value_counts())

# 70/30 split. stratify keeps the same share of habitable planets in both sets,
# which matters because there are so few of them. random_state makes it repeatable.
train, test = train_test_split(labeled, test_size=0.30, random_state=42, stratify=labeled["habitable"])

# the two sets must not share a single planet, otherwise the test score would be meaningless
overlap = set(train["pl_name"]) & set(test["pl_name"])
print("Planets in both sets:", len(overlap))
assert len(overlap) == 0

labeled.to_csv("labeled_sample.csv", index=False)
train.to_csv("train_set.csv", index=False)
test.to_csv("test_set.csv", index=False)
print("Train:", len(train), "habitable:", train["habitable"].sum())
print("Test:", len(test), "habitable:", test["habitable"].sum())

def save_table_image(d, fname, title):
    sample = d.head(6).copy()
    for c in features:
        sample[c] = sample[c].map(lambda v: f"{v:.3g}")
    fig, ax = plt.subplots(figsize=(11, 2.3))
    ax.axis("off")
    ax.set_title(title, loc="left", color=GOLD, fontsize=12, pad=14)
    tbl = ax.table(cellText=sample.values, colLabels=sample.columns, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.5)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor(LINE)
        if row == 0:
            cell.set_facecolor(PANEL); cell.set_text_props(color=GOLD, weight="bold")
        else:
            cell.set_facecolor(BG); cell.set_text_props(color=TEXT)
    plt.savefig(f"charts/{fname}", dpi=150, bbox_inches="tight")
    plt.close()

save_table_image(labeled, "labeled_preview.png", f"Labeled sample ({len(labeled)} planets, {labeled['habitable'].sum()} habitable)")
save_table_image(train, "train_preview.png", f"Training set ({len(train)} planets, {train['habitable'].sum()} habitable)")
save_table_image(test, "test_preview.png", f"Testing set ({len(test)} planets, {test['habitable'].sum()} habitable)")
print("done")

Labeled rows: 5048
habitable
0    4982
1      66
Name: count, dtype: int64
Planets in both sets: 0
Train: 3533 habitable: 46
Test: 1515 habitable: 20
done


## Module 3: Naive Bayes (multinomial) on the labeled exoplanet data.
#### Input:  train_set.csv, test_set.csv (from prep_train_test.py)
#### Output: confusion matrix chart in charts/


In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from matplotlib.colors import LinearSegmentedColormap

BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "font.size": 11,
    "axes.titlesize": 12, "axes.titlecolor": TEXT})

os.makedirs("charts", exist_ok=True)
train = pd.read_csv("train_set.csv")
test = pd.read_csv("test_set.csv")
features = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]

# multinomial NB treats each feature like a count, so every value has to be zero or
# positive. The measurements span several orders of magnitude, so take log10 first,
# then squeeze each column into 0 to 1. The scaler is fit on the training set only,
# and test values that fall outside the training range are clipped to 0 or 1.
X_train = np.log10(train[features])
X_test = np.log10(test[features])
scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = np.clip(scaler.transform(X_test), 0, 1)
y_train, y_test = train["habitable"], test["habitable"]

def report(name, model):
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    print(f"\n{name}")
    print("accuracy:", round(acc, 4), "precision:", round(prec, 4), "recall:", round(rec, 4))
    print(cm)
    return cm, acc

print("Baseline (always guess 'not habitable') accuracy:", round(1 - y_test.mean(), 4))

# model 1: default, the class prior is learned from the training set
nb_default = MultinomialNB().fit(X_train, y_train)
cm1, acc1 = report("MultinomialNB, learned prior", nb_default)

# model 2: same model but both classes get an equal prior
nb_equal = MultinomialNB(fit_prior=False).fit(X_train, y_train)
cm2, acc2 = report("MultinomialNB, equal prior", nb_equal)

cmap = LinearSegmentedColormap.from_list("panel_teal", [PANEL, TEAL])
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, cm, acc, title in [(axes[0], cm1, acc1, "Learned prior"), (axes[1], cm2, acc2, "Equal prior")]:
    ax.imshow(cm, cmap=cmap)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Not habitable", "Habitable"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Not habitable", "Habitable"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{title} (accuracy {acc*100:.1f}%)")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, color=TEXT)
plt.tight_layout()
plt.savefig("charts/nb_confusion_matrices.png", dpi=150)
plt.close()
print("saved charts/nb_confusion_matrices.png")

Baseline (always guess 'not habitable') accuracy: 0.9868

MultinomialNB, learned prior
accuracy: 0.9868 precision: 0.0 recall: 0.0
[[1495    0]
 [  20    0]]

MultinomialNB, equal prior
accuracy: 0.8323 precision: 0.0699 recall: 0.95
[[1242  253]
 [   1   19]]
saved charts/nb_confusion_matrices.png


## Module 3: Decision Trees on the labeled exoplanet data.
#### Input:  train_set.csv, test_set.csv (from prep_train_test.py)
#### Output: three tree diagrams, confusion matrices, and a feature importance chart in charts/


In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from matplotlib.colors import LinearSegmentedColormap

BG, PANEL, LINE = "#0b0f1a", "#131a2b", "#232d47"
TEXT, DIM = "#e9e7e1", "#9aa3bd"
GOLD, TEAL, RUST = "#e8a33d", "#4a95a0", "#c1502e"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": LINE,
    "xtick.color": DIM, "ytick.color": DIM, "font.size": 11,
    "axes.titlesize": 12, "axes.titlecolor": TEXT})

os.makedirs("charts", exist_ok=True)
train = pd.read_csv("train_set.csv")
test = pd.read_csv("test_set.csv")
features = ["pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist"]

# trees split on thresholds, so no scaling or log transform is needed
X_train, y_train = train[features], train["habitable"]
X_test, y_test = test[features], test["habitable"]

print("Baseline (always guess 'not habitable') accuracy:", round(1 - y_test.mean(), 4))

# three different trees: two split rules (gini, entropy) and a third that weights the rare class more
specs = [
    ("Tree 1: Gini, max depth 3", DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)),
    ("Tree 2: Entropy, max depth 3", DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=42)),
    ("Tree 3: Gini, max depth 4, balanced class weights",
     DecisionTreeClassifier(criterion="gini", max_depth=4, class_weight="balanced", random_state=42)),]

def theme_tree(artists):
    # boxes: dark panel; leaves that predict "habitable" are outlined in gold.
    # plot_tree also returns the small True/False edge labels, which have no box,
    # so each artist is judged by its own text instead of its position in the list.
    for a in artists:
        a.set_color(TEXT)
        if a.arrow_patch is not None:
            a.arrow_patch.set_color(DIM)
        patch = a.get_bbox_patch()
        if patch is None:
            continue
        txt = a.get_text()
        is_leaf = "<=" not in txt.split("\n")[0]
        predicts_hab = "class = habitable" in txt
        patch.set_facecolor(PANEL)
        patch.set_edgecolor(GOLD if (is_leaf and predicts_hab) else LINE)
        patch.set_linewidth(2 if (is_leaf and predicts_hab) else 1)

results = []
for i, (name, model) in enumerate(specs, start=1):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    results.append((name, cm, acc))
    print(f"\n{name}")
    print("accuracy:", round(acc, 4), "precision:", round(prec, 4), "recall:", round(rec, 4))
    print(cm)

    fig, ax = plt.subplots(figsize=(14, 7))
    artists = plot_tree(model, feature_names=features, class_names=["not habitable", "habitable"],
                        filled=False, rounded=True, fontsize=8, ax=ax, impurity=True)
    theme_tree(artists)
    ax.set_title(name)
    plt.tight_layout()
    plt.savefig(f"charts/dt_tree_{i}.png", dpi=150)
    plt.close()

# confusion matrices for all three trees side by side
cmap = LinearSegmentedColormap.from_list("panel_teal", [PANEL, TEAL])
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (name, cm, acc) in zip(axes, results):
    ax.imshow(cm, cmap=cmap)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Not hab.", "Habitable"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Not hab.", "Habitable"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{name.split(':')[0]} (accuracy {acc*100:.1f}%)", fontsize=11)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=13, color=TEXT)
plt.tight_layout()
plt.savefig("charts/dt_confusion_matrices.png", dpi=150)
plt.close()

# which measurements the first tree actually used
imp = pd.Series(specs[0][1].feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(imp.index, imp.values, color=TEAL)
ax.set_xlabel("Feature importance")
ax.set_title("Which measurements Tree 1 relies on")
plt.tight_layout()
plt.savefig("charts/dt_feature_importance.png", dpi=150)
plt.close()
print("\nsaved charts")

Baseline (always guess 'not habitable') accuracy: 0.9868

Tree 1: Gini, max depth 3
accuracy: 0.9881 precision: 0.5294 recall: 0.9
[[1479   16]
 [   2   18]]

Tree 2: Entropy, max depth 3
accuracy: 0.9894 precision: 0.5833 recall: 0.7
[[1485   10]
 [   6   14]]

Tree 3: Gini, max depth 4, balanced class weights
accuracy: 0.9848 precision: 0.4634 recall: 0.95
[[1473   22]
 [   1   19]]

saved charts


In [16]:
# Downloading my files!
import shutil
shutil.make_archive('charts', 'zip', 'charts')
from google.colab import files
files.download('charts.zip')
files.download('exoplanets_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>